# Exercise 4 — Bollinger Bands

Bollinger Bands place a volatility envelope around price. The middle band is the SMA; the upper and lower bands are `num_std` standard deviations above and below. Prices touching the upper band may be overbought; touching the lower band may be oversold. Band width narrows in quiet markets and widens in volatile ones.

In [ ]:
import pandas as pd, math

def _synthetic(n=50):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
def sma(series, window=20):
    return series.rolling(window=window).mean()
def ema(series, window=20):
    return series.ewm(span=window, adjust=False).mean()
def rsi(series, window=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(window=window).mean()
    loss  = (-delta.clip(upper=0)).rolling(window=window).mean()
    rs    = gain / loss
    return 100 - (100 / (1 + rs))
def macd(series, fast=12, slow=26, signal=9):
    fast_ema    = ema(series, fast)
    slow_ema    = ema(series, slow)
    macd_line   = fast_ema - slow_ema
    signal_line = ema(macd_line, signal)
    return pd.DataFrame({"macd": macd_line, "signal": signal_line,
                          "histogram": macd_line - signal_line})

# ── Exercise: implement bollinger_bands ──────────────────────────────────────

def bollinger_bands(series, window=20, num_std=2.0):
    """Bollinger Bands.

    Args:
        series  : pd.Series of prices
        window  : SMA and rolling-std look-back (default 20)
        num_std : number of standard deviations for the bands (default 2.0)

    Returns:
        pd.DataFrame with columns:
            upper  — SMA + num_std * rolling_std
            middle — SMA
            lower  — SMA - num_std * rolling_std
        First window-1 rows are NaN.
    """
    # TODO:
    # 1. middle = sma(series, window)
    # 2. std    = series.rolling(window=window).std()
    # 3. upper  = middle + num_std * std
    # 4. lower  = middle - num_std * std
    # 5. return pd.DataFrame({"upper": upper, "middle": middle, "lower": lower})
    n = len(series)
    return pd.DataFrame({"upper":  [float("nan")] * n,
                          "middle": [float("nan")] * n,
                          "lower":  [float("nan")] * n},
                         index=series.index)


### Checks

In [ ]:
checks = 0

# 1 — bollinger_bands returns DataFrame with 3 columns
try:
    close = _synthetic()["Close"]
    bb = bollinger_bands(close, 20)
    assert isinstance(bb, pd.DataFrame)
    assert "upper" in bb.columns and "middle" in bb.columns and "lower" in bb.columns
    assert len(bb) == len(close)
    checks += 1; print("✅ 1 bollinger_bands returns DataFrame with upper/middle/lower")
except Exception as e:
    print("❌ 1:", e)

# 2 — first window-1 rows are NaN; row window-1 is not NaN
try:
    close = _synthetic()["Close"]
    bb = bollinger_bands(close, 20)
    assert bb["middle"].iloc[:19].isna().all(), "first 19 should be NaN"
    assert not pd.isna(bb["middle"].iloc[19]), "index 19 should be first non-NaN"
    checks += 1; print("✅ 2 first window-1 rows are NaN")
except Exception as e:
    print("❌ 2:", e)

# 3 — upper > middle > lower for all non-NaN rows
try:
    close = _synthetic()["Close"]
    bb = bollinger_bands(close, 20)
    non_nan = bb.dropna()
    assert (non_nan["upper"] > non_nan["middle"]).all(), "upper <= middle in some rows"
    assert (non_nan["middle"] > non_nan["lower"]).all(), "middle <= lower in some rows"
    checks += 1; print("✅ 3 upper > middle > lower for all non-NaN rows")
except Exception as e:
    print("❌ 3:", e)

# 4 — middle equals SMA of close with window
try:
    close = _synthetic()["Close"]
    bb = bollinger_bands(close, 10)
    s   = sma(close, 10)
    diff = (bb["middle"] - s).dropna().abs().max()
    assert diff < 1e-9, f"middle != sma: max diff={diff}"
    checks += 1; print("✅ 4 middle band equals SMA with same window")
except Exception as e:
    print("❌ 4:", e)

# 5 — wider bands for more volatile price series
try:
    volatile = pd.Series([100.0 + 10.0 * ((-1)**i) for i in range(50)])
    stable   = pd.Series([100.0 + 0.1 * ((-1)**i)  for i in range(50)])
    bb_v = bollinger_bands(volatile, 10)
    bb_s = bollinger_bands(stable,   10)
    width_v = (bb_v["upper"] - bb_v["lower"]).dropna().mean()
    width_s = (bb_s["upper"] - bb_s["lower"]).dropna().mean()
    assert width_v > width_s, f"volatile width ({width_v:.2f}) should > stable ({width_s:.2f})"
    checks += 1; print("✅ 5 bands are wider for more volatile price series")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
